# BharatCrop Intelligence
## Notebook 02 — Dataset 1: Historical APY Acquisition and Audit

### Objective
Acquire and validate district-level agricultural statistics for Food Grains
from agricultural year 2013-14 through 2024-25.

### Source
Department of Agriculture & Farmers Welfare — UPAg portal.

### Workflow
1. Locate the project and source files.
2. Load the four original CSV exports.
3. Validate their schemas, years, data types, and identifier keys.
4. Merge the four batches into one historical dataset.
5. Validate the merged result and document data-quality issues.
6. Save the merged dataset and audit reports without modifying raw files.

### Important rules
- Preserve the original downloaded CSV files.
- Do not impute missing values without understanding their meaning.
- Do not silently remove duplicate records or aggregate crop categories.
- Record validation failures rather than hiding them.


In [1]:

# Import the libraries used throughout this notebook.
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Locate the project root.
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

# Define the folders used by this notebook.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AUDIT_DIR = PROJECT_ROOT / "reports" / "data_audit"
METADATA_DIR = PROJECT_ROOT / "metadata"

# Create output folders if they do not already exist.
for folder in [PROCESSED_DIR, AUDIT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data folder:", RAW_DIR.resolve())
print("Processed data folder:", PROCESSED_DIR.resolve())
print("Audit reports folder:", AUDIT_DIR.resolve())


Project root: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence
Raw data folder: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\raw
Processed data folder: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed
Audit reports folder: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\reports\data_audit


In [2]:

# List the four downloaded batches in chronological order.
batch_files = [
    RAW_DIR / "DES-District-Data-For-2013-14-to-2015-16.csv",
    RAW_DIR / "DES-District-Data-For-2016-17-to-2018-19.csv",
    RAW_DIR / "DES-District-Data-For-2019-20-to-2021-22.csv",
    RAW_DIR / "DES-District-Data-For-2022-23-to-2024-25.csv",
]

# Check that every required file exists before reading anything.
missing_files = [path for path in batch_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "The following files are missing:\n"
        + "\n".join(str(path) for path in missing_files)
    )

print("All four historical APY files were found.\n")

for path in batch_files:
    print(f"{path.name} | {path.stat().st_size / (1024 ** 2):.2f} MB")


All four historical APY files were found.

DES-District-Data-For-2013-14-to-2015-16.csv | 2.49 MB
DES-District-Data-For-2016-17-to-2018-19.csv | 2.71 MB
DES-District-Data-For-2019-20-to-2021-22.csv | 2.86 MB
DES-District-Data-For-2022-23-to-2024-25.csv | 3.07 MB


In [3]:

# Read the CSV exports into separate DataFrames.
# Keeping them separate lets us inspect and validate each batch.
apy_batches = {}

for path in batch_files:
    apy_batches[path.name] = pd.read_csv(path)

# Display a concise summary of every batch.
batch_summary = []

for filename, df in apy_batches.items():
    batch_summary.append({
        "filename": filename,
        "rows": len(df),
        "columns": len(df.columns),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
    })

batch_summary = pd.DataFrame(batch_summary)

display(batch_summary)

# Inspect a sample from the first batch.
display(next(iter(apy_batches.values())).head())


,filename,rows,columns,exact_duplicate_rows,missing_cells
0,DES-District-Data-For-2013-14-to-2015-16.csv,23007,13,0,15977
1,DES-District-Data-For-2016-17-to-2018-19.csv,24914,13,0,16260
2,DES-District-Data-For-2019-20-to-2021-22.csv,26413,13,0,19845
3,DES-District-Data-For-2022-23-to-2024-25.csv,28661,13,0,30303


,State,District,Crop,Season,Area-2013-14,Area-2014-15,Area-2015-16,Production-2013-14,Production-2014-15,Production-2015-16,Yield-2013-14,Yield-2014-15,Yield-2015-16
0,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,0.0,0.0,0.0,0.0,0.0,0.0,2377.0,2348.0,2240.0
1,Andaman And Nicobar Islands,Nicobars,Rice,Total,0.0,0.0,0.0,0.0,0.0,0.0,2377.0,2348.0,2240.0
2,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,0.0,0.0,0.0,0.0,0.0,0.0,526.0,9167.0,1000.0
3,Andaman And Nicobar Islands,Nicobars,Maize,Total,0.0,0.0,0.0,0.0,0.0,0.0,526.0,9167.0,1000.0
4,Andaman And Nicobar Islands,Nicobars,Nutri/Coarse Cereals,Rabi,0.0,0.0,0.0,0.0,0.0,0.0,526.0,9167.0,1000.0


In [4]:

# The columns that identify each crop observation.
key_columns = ["State", "District", "Crop", "Season"]

# The expected agricultural years in each export.
batch_years = {
    "DES-District-Data-For-2013-14-to-2015-16.csv":
        ["2013-14", "2014-15", "2015-16"],

    "DES-District-Data-For-2016-17-to-2018-19.csv":
        ["2016-17", "2017-18", "2018-19"],

    "DES-District-Data-For-2019-20-to-2021-22.csv":
        ["2019-20", "2020-21", "2021-22"],

    "DES-District-Data-For-2022-23-to-2024-25.csv":
        ["2022-23", "2023-24", "2024-25"],
}

metrics = ["Area", "Production", "Yield"]

schema_results = []

for filename, df in apy_batches.items():
    expected_columns = key_columns + [
        f"{metric}-{year}"
        for year in batch_years[filename]
        for metric in metrics
    ]

    missing_columns = [
        col for col in expected_columns if col not in df.columns
    ]

    unexpected_columns = [
        col for col in df.columns if col not in expected_columns
    ]

    schema_results.append({
        "filename": filename,
        "expected_column_count": len(expected_columns),
        "actual_column_count": len(df.columns),
        "missing_columns": missing_columns,
        "unexpected_columns": unexpected_columns,
        "schema_valid": (
            not missing_columns and not unexpected_columns
        ),
    })

schema_report = pd.DataFrame(schema_results)
display(schema_report)

if not schema_report["schema_valid"].all():
    raise ValueError(
        "At least one batch has a different schema. "
        "Inspect schema_report before merging."
    )

print("All four batch schemas match their expected year ranges.")


,filename,expected_column_count,actual_column_count,missing_columns,unexpected_columns,schema_valid
0,DES-District-Data-For-2013-14-to-2015-16.csv,13,13,[],[],True
1,DES-District-Data-For-2016-17-to-2018-19.csv,13,13,[],[],True
2,DES-District-Data-For-2019-20-to-2021-22.csv,13,13,[],[],True
3,DES-District-Data-For-2022-23-to-2024-25.csv,13,13,[],[],True


All four batch schemas match their expected year ranges.


In [5]:

key_validation_results = []

for filename, df in apy_batches.items():
    # Missing identifiers make records unreliable for joining.
    missing_key_rows = int(df[key_columns].isna().any(axis=1).sum())

    # Count every row belonging to a repeated identifier combination.
    repeated_key_rows = int(
        df.duplicated(subset=key_columns, keep=False).sum()
    )

    key_validation_results.append({
        "filename": filename,
        "rows": len(df),
        "missing_key_rows": missing_key_rows,
        "rows_with_repeated_keys": repeated_key_rows,
        "keys_unique": repeated_key_rows == 0,
        "keys_complete": missing_key_rows == 0,
    })

key_validation_report = pd.DataFrame(key_validation_results)
display(key_validation_report)

if not key_validation_report["keys_unique"].all():
    raise ValueError(
        "Repeated identifier keys detected. Inspect the affected "
        "batch before merging; do not automatically drop duplicates."
    )

if not key_validation_report["keys_complete"].all():
    raise ValueError(
        "Missing identifier values detected. Resolve or document "
        "these records before merging."
    )

print("All batches have complete, unique identifier keys.")


,filename,rows,missing_key_rows,rows_with_repeated_keys,keys_unique,keys_complete
0,DES-District-Data-For-2013-14-to-2015-16.csv,23007,0,0,True,True
1,DES-District-Data-For-2016-17-to-2018-19.csv,24914,0,0,True,True
2,DES-District-Data-For-2019-20-to-2021-22.csv,26413,0,0,True,True
3,DES-District-Data-For-2022-23-to-2024-25.csv,28661,0,0,True,True


All batches have complete, unique identifier keys.


In [6]:

# Start with the earliest batch and merge each subsequent batch.
batch_names = list(apy_batches.keys())

apy_historical = apy_batches[batch_names[0]].copy()

for filename in batch_names[1:]:
    next_batch = apy_batches[filename].copy()

    # An outer, one-to-one merge prevents silent row multiplication.
    apy_historical = apy_historical.merge(
        next_batch,
        on=key_columns,
        how="outer",
        validate="one_to_one",
        indicator="_merge_status",
    )

    # Report whether observations appear in both batches or only one.
    print(f"\nMerged batch: {filename}")
    display(
        apy_historical["_merge_status"]
        .value_counts(dropna=False)
        .rename_axis("merge_status")
        .to_frame("row_count")
    )

    # The indicator belongs only to this merge operation.
    apy_historical = apy_historical.drop(columns="_merge_status")

# Arrange the columns in a predictable order.
year_sequence = [
    "2013-14", "2014-15", "2015-16",
    "2016-17", "2017-18", "2018-19",
    "2019-20", "2020-21", "2021-22",
    "2022-23", "2023-24", "2024-25",
]

measure_columns = [
    f"{metric}-{year}"
    for year in year_sequence
    for metric in metrics
]

apy_historical = apy_historical[key_columns + measure_columns]

# Sort only for readability; this does not aggregate or remove records.
apy_historical = apy_historical.sort_values(
    key_columns
).reset_index(drop=True)

print("\nMerged historical dataset shape:", apy_historical.shape)
display(apy_historical.head())



Merged batch: DES-District-Data-For-2016-17-to-2018-19.csv


,row_count
merge_status,
both,22021
right_only,2893
left_only,986



Merged batch: DES-District-Data-For-2019-20-to-2021-22.csv


,row_count
merge_status,
both,24260
right_only,2153
left_only,1640



Merged batch: DES-District-Data-For-2022-23-to-2024-25.csv


,row_count
merge_status,
both,25358
right_only,3303
left_only,2695



Merged historical dataset shape: (31356, 40)


,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14,Area-2014-15,Production-2014-15,Yield-2014-15,...,Yield-2021-22,Area-2022-23,Production-2022-23,Yield-2022-23,Area-2023-24,Production-2023-24,Yield-2023-24,Area-2024-25,Production-2024-25,Yield-2024-25
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,2377.0,0.0,0.0,2348.0,...,NaN,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.0,0.0,923.0,0.0,0.0,3135.0,...,1133.0,0.0,0.0,1260.0,0.0,0.0,2266.0,0.0,0.0,735.0
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,0.0,0.0,526.0,0.0,0.0,9167.0,...,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,0.0,0.0,526.0,0.0,0.0,9167.0,...,1133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:

# Check the merged result without changing its values.
numeric_columns = [
    col for col in apy_historical.columns
    if col not in key_columns
]

missing_report = pd.DataFrame({
    "missing_count": apy_historical.isna().sum(),
    "missing_percent": (
        apy_historical.isna().mean() * 100
    ).round(3),
})

print("Final shape:", apy_historical.shape)
print("Exact duplicate rows:", apy_historical.duplicated().sum())
print(
    "Repeated identifier keys:",
    apy_historical.duplicated(subset=key_columns, keep=False).sum()
)

print("\nUnique states:", apy_historical["State"].nunique())
print(
    "Unique state-district pairs:",
    apy_historical[["State", "District"]].drop_duplicates().shape[0]
)
print("Unique crop labels:", apy_historical["Crop"].nunique())
print("Season counts:")
display(apy_historical["Season"].value_counts(dropna=False).to_frame("rows"))

print("\nMissing values by column:")
display(missing_report)

print("\nNumeric summary:")
display(apy_historical[numeric_columns].describe().T)

print("\nNegative numeric values:")
display((apy_historical[numeric_columns] < 0).sum().to_frame("negative_count"))


Final shape: (31356, 40)
Exact duplicate rows: 0
Repeated identifier keys: 0

Unique states: 34
Unique state-district pairs: 751
Unique crop labels: 19
Season counts:


,rows
Season,
Total,11919
Kharif,8891
Rabi,7392
Summer,3154



Missing values by column:


,missing_count,missing_percent
State,0,0.000
District,0,0.000
Crop,0,0.000
Season,0,0.000
Area-2013-14,10428,33.257
Production-2013-14,10548,33.639
Yield-2013-14,10548,33.639
Area-2014-15,10001,31.895
Production-2014-15,10152,32.377
Yield-2014-15,10152,32.377



Numeric summary:


,count,mean,std,min,25%,50%,75%,max
Area-2013-14,20928.0,0.377688,0.807904,0.0,0.00,0.03,0.34,11.63
Production-2013-14,20808.0,0.846260,2.023841,0.0,0.00,0.04,0.56,28.70
Yield-2013-14,20808.0,1665.355873,1225.886281,2.0,777.00,1285.50,2300.00,11375.00
Area-2014-15,21355.0,0.368406,0.776899,0.0,0.00,0.04,0.33,9.94
Production-2014-15,21204.0,0.782656,1.858980,0.0,0.00,0.04,0.55,26.63
Yield-2014-15,21204.0,1634.272684,1222.958686,0.0,760.00,1330.00,2188.00,11292.00
Area-2015-16,21623.0,0.360860,0.771538,0.0,0.00,0.03,0.31,10.81
Production-2015-16,21578.0,0.734970,1.839879,0.0,0.00,0.04,0.45,27.20
Yield-2015-16,21578.0,1570.087357,1221.702343,0.0,701.00,1192.00,2178.00,11978.00
Area-2016-17,22489.0,0.367157,0.803707,0.0,0.00,0.03,0.32,11.51



Negative numeric values:


,negative_count
Area-2013-14,0
Production-2013-14,0
Yield-2013-14,0
Area-2014-15,0
Production-2014-15,0
Yield-2014-15,0
Area-2015-16,0
Production-2015-16,0
Yield-2015-16,0
Area-2016-17,0


In [8]:

# Output paths for processed data and audit evidence.
merged_path = (
    PROCESSED_DIR / "historical_food_grains_apy_2013-14_to_2024-25.csv"
)
missing_report_path = AUDIT_DIR / "historical_apy_missing_values.csv"
batch_report_path = AUDIT_DIR / "historical_apy_batch_summary.csv"
manifest_path = AUDIT_DIR / "historical_apy_manifest.json"

# Save the merged data and reports.
apy_historical.to_csv(merged_path, index=False)
missing_report.to_csv(missing_report_path)
batch_summary.to_csv(batch_report_path, index=False)

manifest = {
    "dataset_name": "Historical district-level APY — Food Grains",
    "source": "UPAg / Department of Agriculture & Farmers Welfare",
    "years": year_sequence,
    "source_files": [path.name for path in batch_files],
    "merge_keys": key_columns,
    "merged_rows": int(len(apy_historical)),
    "merged_columns": int(len(apy_historical.columns)),
    "exact_duplicate_rows": int(apy_historical.duplicated().sum()),
    "repeated_key_rows": int(
        apy_historical.duplicated(
            subset=key_columns, keep=False
        ).sum()
    ),
    "missing_cells": int(apy_historical.isna().sum().sum()),
    "raw_files_modified": False,
}

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

print("Merged dataset saved to:", merged_path.resolve())
print("Missing-value report saved to:", missing_report_path.resolve())
print("Batch summary saved to:", batch_report_path.resolve())
print("Manifest saved to:", manifest_path.resolve())

print("\nAcquisition and merge workflow completed.")
print("Remember: missing values and unusual records have not been imputed or removed.")


Merged dataset saved to: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\data\processed\historical_food_grains_apy_2013-14_to_2024-25.csv
Missing-value report saved to: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\reports\data_audit\historical_apy_missing_values.csv
Batch summary saved to: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\reports\data_audit\historical_apy_batch_summary.csv
Manifest saved to: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\reports\data_audit\historical_apy_manifest.json

Acquisition and merge workflow completed.
Remember: missing values and unusual records have not been imputed or removed.


In [9]:

# ------------------------------------------------------------
# Cell 10: Investigate coverage differences and suspicious values
# ------------------------------------------------------------

# 1. Compare the observation keys across all four source batches.
key_sets = {
    filename: set(map(tuple, df[key_columns].itertuples(index=False, name=None)))
    for filename, df in apy_batches.items()
}

coverage_rows = []

for filename, keys in key_sets.items():
    coverage_rows.append({
        "batch": filename,
        "unique_keys": len(keys),
        "keys_in_all_other_batches": len(
            keys.intersection(
                set.intersection(
                    *(other for name, other in key_sets.items()
                      if name != filename)
                )
            )
        ),
    })

display(pd.DataFrame(coverage_rows))

# 2. Show examples of keys that are not present in every batch.
all_key_sets = list(key_sets.items())

for filename, keys in all_key_sets:
    other_keys = set.union(*[
        other_keys
        for other_name, other_keys in all_key_sets
        if other_name != filename
    ])

    keys_unique_to_batch = keys - other_keys

    print(f"\nKeys found only in {filename}: {len(keys_unique_to_batch)}")
    display(
        pd.DataFrame(
            list(keys_unique_to_batch),
            columns=key_columns
        ).head(10)
    )

# 3. Compare missing values in each raw batch with the merged dataset.
raw_missing_rows = []

for filename, df in apy_batches.items():
    for column in df.columns:
        raw_missing_rows.append({
            "batch": filename,
            "column": column,
            "missing_count": int(df[column].isna().sum()),
            "missing_percent": round(df[column].isna().mean() * 100, 3),
        })

raw_missing_report = pd.DataFrame(raw_missing_rows)

print("\nRaw-file missingness summary:")
display(
    raw_missing_report[
        raw_missing_report["missing_count"] > 0
    ].sort_values(
        ["batch", "missing_count"],
        ascending=[True, False]
    )
)

# 4. Inspect the most extreme yield records by year.
yield_columns = [
    f"Yield-{year}"
    for year in year_sequence
]

for column in yield_columns:
    numeric_yield = pd.to_numeric(
        apy_historical[column],
        errors="coerce"
    )

    valid_yield = numeric_yield.dropna()

    if valid_yield.empty:
        continue

    print(f"\nHighest 10 values for {column}:")
    extreme_indices = valid_yield.nlargest(10).index

    display(
        apy_historical.loc[
            extreme_indices,
            key_columns + [column]
        ].sort_values(column, ascending=False)
    )

print("\nInvestigation complete. No records were modified.")


,batch,unique_keys,keys_in_all_other_batches
0,DES-District-Data-For-2013-14-to-2015-16.csv,23007,20832
1,DES-District-Data-For-2016-17-to-2018-19.csv,24914,20832
2,DES-District-Data-For-2019-20-to-2021-22.csv,26413,20832
3,DES-District-Data-For-2022-23-to-2024-25.csv,28661,20832



Keys found only in DES-District-Data-For-2013-14-to-2015-16.csv: 496


,State,District,Crop,Season
0,Nagaland,Peren,Urad,Kharif
1,Assam,Dima Hasao,Rice,Summer
2,Madhya Pradesh,Jhabua,Shree Anna /Nutri Cereals,Rabi
3,Tamil Nadu,Dindigul,Lentil,Total
4,Maharashtra,Gondia,Small Millets,Rabi
5,Madhya Pradesh,Sagar,Ragi,Kharif
6,Karnataka,Dakshina Kannada,Nutri/Coarse Cereals,Summer
7,Madhya Pradesh,Ratlam,Jowar,Rabi
8,Madhya Pradesh,Damoh,Urad,Rabi
9,Goa,North Goa,Ragi,Kharif



Keys found only in DES-District-Data-For-2016-17-to-2018-19.csv: 204


,State,District,Crop,Season
0,Madhya Pradesh,Balaghat,Bajra,Total
1,Madhya Pradesh,Bhind,Small Millets,Total
2,Rajasthan,Bikaner,Urad,Total
3,Jammu And Kashmir,Udhampur,Ragi,Kharif
4,Jammu And Kashmir,Udhampur,Ragi,Total
5,Andhra Pradesh,Visakhapatnam,Gram,Kharif
6,Rajasthan,Rajsamand,Small Millets,Kharif
7,Rajasthan,Rajsamand,Small Millets,Total
8,Punjab,Faridkot,Tur,Kharif
9,Madhya Pradesh,Shajapur,Small Millets,Total



Keys found only in DES-District-Data-For-2019-20-to-2021-22.csv: 715


... (26 outputs left out)

,State,District,Crop,Season,Yield-2023-24
64,Andaman And Nicobar Islands,South Andamans,Urad,Rabi,385540.0
65,Andaman And Nicobar Islands,South Andamans,Urad,Total,385540.0
59,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,161579.0
61,Andaman And Nicobar Islands,South Andamans,Total Pulses,Total,161579.0
49,Andaman And Nicobar Islands,South Andamans,Moong,Rabi,105588.0
50,Andaman And Nicobar Islands,South Andamans,Moong,Total,105588.0
21391,Rajasthan,Bikaner,Other Pulses,Rabi,20000.0
21769,Rajasthan,Hanumangarh,Other Pulses,Rabi,14889.0
13058,Kerala,Thrissur,Jowar,Rabi,12687.0
13059,Kerala,Thrissur,Jowar,Total,12687.0



Highest 10 values for Yield-2024-25:


,State,District,Crop,Season,Yield-2024-25
21391,Rajasthan,Bikaner,Other Pulses,Rabi,15033.0
25393,Telangana,Nalgonda,Other Pulses,Rabi,13717.0
30613,West Bengal,Dakshin Dinajpur,Maize,Summer,12237.0
30614,West Bengal,Dakshin Dinajpur,Maize,Total,12237.0
30619,West Bengal,Dakshin Dinajpur,Nutri/Coarse Cereals,Summer,12237.0
30620,West Bengal,Dakshin Dinajpur,Nutri/Coarse Cereals,Total,12237.0
25394,Telangana,Nalgonda,Other Pulses,Total,11842.0
30909,West Bengal,Malda,Maize,Rabi,11295.0
30917,West Bengal,Malda,Nutri/Coarse Cereals,Rabi,11295.0
21476,Rajasthan,Chittorgarh,Other Pulses,Rabi,11204.0



Investigation complete. No records were modified.


In [10]:

# Cell 11: Investigate yield anomalies and aggregate crop categories.
# This is a diagnostic check only. It does not change the dataset.

import numpy as np
import pandas as pd

merged_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "historical_food_grains_apy_2013-14_to_2024-25.csv"
)

apy = pd.read_csv(merged_path)

# Identify the agricultural years represented in the merged dataset.
years = [
    f"{start}-{str(start + 1)[-2:]}"
    for start in range(2013, 2025)
]

# Display extreme yields with their corresponding area and production.
# This helps us investigate unusual records instead of deleting them blindly.
for year in years:
    area_col = f"Area-{year}"
    production_col = f"Production-{year}"
    yield_col = f"Yield-{year}"

    if not all(
        col in apy.columns
        for col in [area_col, production_col, yield_col]
    ):
        print(f"Skipping {year}: expected columns are missing.")
        continue

    extreme = (
        apy[
            [
                "State",
                "District",
                "Crop",
                "Season",
                area_col,
                production_col,
                yield_col,
            ]
        ]
        .dropna(subset=[yield_col])
        .sort_values(yield_col, ascending=False)
        .head(5)
    )

    print(f"\nTop 5 reported yields for {year}")
    display(extreme)

# Count rows belonging to aggregate categories.
aggregate_labels = [
    "Cereals",
    "Nutri/Coarse Cereals",
    "Shree Anna /Nutri Cereals",
    "Total Pulses",
    "Total Food Grains",
]

aggregate_summary = (
    apy[apy["Crop"].isin(aggregate_labels)]
    ["Crop"]
    .value_counts()
    .rename_axis("crop_category")
    .reset_index(name="row_count")
)

print("\nAggregate crop category counts")
display(aggregate_summary)

# Check whether area and production are both zero while yield is positive.
# Such records deserve investigation; they are not automatically invalid.
for year in years:
    area_col = f"Area-{year}"
    production_col = f"Production-{year}"
    yield_col = f"Yield-{year}"

    if not all(
        col in apy.columns
        for col in [area_col, production_col, yield_col]
    ):
        continue

    suspicious = apy[
        apy[area_col].eq(0)
        & apy[production_col].eq(0)
        & apy[yield_col].gt(0)
    ]

    print(
        f"{year}: rows with zero area, zero production, "
        f"and positive yield = {len(suspicious):,}"
    )

print("\nDiagnostic checks complete. No records were modified.")



Top 5 reported yields for 2013-14


,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14
21770,Rajasthan,Hanumangarh,Other Pulses,Total,0.0,0.00,11375.0
21769,Rajasthan,Hanumangarh,Other Pulses,Rabi,0.0,0.00,11375.0
22492,Rajasthan,Sirohi,Other Pulses,Rabi,0.0,0.00,9400.0
22493,Rajasthan,Sirohi,Other Pulses,Total,0.0,0.00,9400.0
8186,Haryana,Fatehabad,Other Pulses,Kharif,0.0,0.03,9059.0



Top 5 reported yields for 2014-15


,State,District,Crop,Season,Area-2014-15,Production-2014-15,Yield-2014-15
23554,Tamil Nadu,Perambalur,Maize,Kharif,0.34,3.88,11292.0
21770,Rajasthan,Hanumangarh,Other Pulses,Total,0.00,0.00,11227.0
21769,Rajasthan,Hanumangarh,Other Pulses,Rabi,0.00,0.00,11227.0
23556,Tamil Nadu,Perambalur,Maize,Total,0.37,4.06,11036.0
23560,Tamil Nadu,Perambalur,Nutri/Coarse Cereals,Kharif,0.36,3.91,10753.0



Top 5 reported yields for 2015-16


,State,District,Crop,Season,Area-2015-16,Production-2015-16,Yield-2015-16
24308,Tamil Nadu,Tiruppur,Maize,Kharif,0.06,0.71,11978.0
23747,Tamil Nadu,Salem,Maize,Kharif,0.25,2.76,11162.0
8428,Haryana,Kurukshetra,Other Pulses,Kharif,0.02,0.17,10844.0
4063,Bihar,Katihar,Maize,Rabi,0.14,1.41,10301.0
4070,Bihar,Katihar,Nutri/Coarse Cereals,Rabi,0.14,1.41,10284.0


... (17 outputs left out)

,State,District,Crop,Season,Area-2024-25,Production-2024-25,Yield-2024-25
21391,Rajasthan,Bikaner,Other Pulses,Rabi,0.00,0.00,15033.0
25393,Telangana,Nalgonda,Other Pulses,Rabi,0.00,0.01,13717.0
30614,West Bengal,Dakshin Dinajpur,Maize,Total,0.02,0.29,12237.0
30620,West Bengal,Dakshin Dinajpur,Nutri/Coarse Cereals,Total,0.02,0.29,12237.0
30619,West Bengal,Dakshin Dinajpur,Nutri/Coarse Cereals,Summer,0.02,0.29,12237.0



Aggregate crop category counts


,crop_category,row_count
0,Total Food Grains,2759
1,Cereals,2677
2,Nutri/Coarse Cereals,2530
3,Shree Anna /Nutri Cereals,1803
4,Total Pulses,747


2013-14: rows with zero area, zero production, and positive yield = 5,296
2014-15: rows with zero area, zero production, and positive yield = 5,068
2015-16: rows with zero area, zero production, and positive yield = 5,338
2016-17: rows with zero area, zero production, and positive yield = 5,768
2017-18: rows with zero area, zero production, and positive yield = 6,011
2018-19: rows with zero area, zero production, and positive yield = 6,700
2019-20: rows with zero area, zero production, and positive yield = 6,993
2020-21: rows with zero area, zero production, and positive yield = 7,193
2021-22: rows with zero area, zero production, and positive yield = 7,025
2022-23: rows with zero area, zero production, and positive yield = 7,471
2023-24: rows with zero area, zero production, and positive yield = 7,805
2024-25: rows with zero area, zero production, and positive yield = 7,682

Diagnostic checks complete. No records were modified.


In [11]:

# Cell 12: Compare reported yield with yield implied by area and production.
# Assumption to verify: area is in lakh hectares and production in lakh tonnes.
# Under that assumption, yield in kg/ha = production / area * 1000.

yield_check_rows = []

for year in year_sequence:
    area_col = f"Area-{year}"
    production_col = f"Production-{year}"
    yield_col = f"Yield-{year}"

    area = pd.to_numeric(apy_historical[area_col], errors="coerce")
    production = pd.to_numeric(
        apy_historical[production_col], errors="coerce"
    )
    reported_yield = pd.to_numeric(
        apy_historical[yield_col], errors="coerce"
    )

    valid = (
        area.gt(0)
        & production.notna()
        & reported_yield.notna()
    )

    implied_yield = (production[valid] / area[valid]) * 1000

    comparison = pd.DataFrame({
        "reported_yield": reported_yield[valid],
        "implied_yield": implied_yield,
    })

    comparison["absolute_difference"] = (
        comparison["reported_yield"] - comparison["implied_yield"]
    ).abs()

    comparison["relative_difference_pct"] = (
        comparison["absolute_difference"]
        / comparison["reported_yield"].replace(0, np.nan)
        * 100
    )

    zero_area_positive_yield = int(
        (area.eq(0) & reported_yield.gt(0)).sum()
    )

    yield_check_rows.append({
        "year": year,
        "records_checked": int(valid.sum()),
        "median_absolute_difference": comparison[
            "absolute_difference"
        ].median(),
        "median_relative_difference_pct": comparison[
            "relative_difference_pct"
        ].median(),
        "zero_area_positive_yield_rows": zero_area_positive_yield,
    })

yield_consistency_report = pd.DataFrame(yield_check_rows)
display(yield_consistency_report)

print(
    "Interpretation: this is a diagnostic under the stated unit assumption, "
    "not proof that the source uses those units."
)


,year,records_checked,median_absolute_difference,median_relative_difference_pct,zero_area_positive_yield_rows
0,2013-14,14910,23.000000,1.601093,5898
1,2014-15,15574,17.333333,1.205762,5622
2,2015-16,15612,19.924300,1.471334,5931
3,2016-17,16001,23.000000,1.487177,6423
4,2017-18,16519,29.000000,1.823282,6795
5,2018-19,16023,28.500000,1.892285,7469
6,2019-20,16325,31.000000,1.960784,7830
7,2020-21,16235,29.666667,1.791531,8101
8,2021-22,16052,30.257453,1.851852,7921
9,2022-23,16633,31.363636,1.843911,8389


Interpretation: this is a diagnostic under the stated unit assumption, not proof that the source uses those units.


In [12]:
# Cell 13: Display the yield-consistency audit with explicit column names.
# This makes it easier to verify that each result is in the correct column.

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print("Yield consistency report")
display(
    yield_consistency_report
    .rename_axis(None)
    .reset_index(drop=True)
    .round(4)
)

print("\nData types")
display(yield_consistency_report.dtypes.rename("dtype").to_frame())

print("\nCheck for missing diagnostic results")
display(yield_consistency_report.isna().sum().rename("missing_count").to_frame())

Yield consistency report


,year,records_checked,median_absolute_difference,median_relative_difference_pct,zero_area_positive_yield_rows
0,2013-14,14910,23.0000,1.6011,5898
1,2014-15,15574,17.3333,1.2058,5622
2,2015-16,15612,19.9243,1.4713,5931
3,2016-17,16001,23.0000,1.4872,6423
4,2017-18,16519,29.0000,1.8233,6795
5,2018-19,16023,28.5000,1.8923,7469
6,2019-20,16325,31.0000,1.9608,7830
7,2020-21,16235,29.6667,1.7915,8101
8,2021-22,16052,30.2575,1.8519,7921
9,2022-23,16633,31.3636,1.8439,8389



Data types


,dtype
year,object
records_checked,int64
median_absolute_difference,float64
median_relative_difference_pct,float64
zero_area_positive_yield_rows,int64



Check for missing diagnostic results


,missing_count
year,0
records_checked,0
median_absolute_difference,0
median_relative_difference_pct,0
zero_area_positive_yield_rows,0


In [13]:
# Cell 14: Investigate zero-area, positive-yield records.
# These are quality flags for investigation, not automatic deletion rules.

zero_area_positive_yield_summary = []

for year in year_sequence:
    area_col = f"Area-{year}"
    yield_col = f"Yield-{year}"

    area = pd.to_numeric(apy_historical[area_col], errors="coerce")
    reported_yield = pd.to_numeric(
        apy_historical[yield_col], errors="coerce"
    )

    mask = area.eq(0) & reported_yield.gt(0)

    flagged = apy_historical.loc[
        mask, ["State", "District", "Crop", "Season"]
    ].copy()

    zero_area_positive_yield_summary.append({
        "year": year,
        "flagged_rows": len(flagged),
        "unique_crops": flagged["Crop"].nunique(),
        "unique_districts": flagged["District"].nunique(),
        "top_crops": flagged["Crop"].value_counts().head(5).to_dict(),
        "top_seasons": flagged["Season"].value_counts().head(5).to_dict(),
    })

zero_area_positive_yield_report = pd.DataFrame(
    zero_area_positive_yield_summary
)

display(zero_area_positive_yield_report)

,year,flagged_rows,unique_crops,unique_districts,top_crops,top_seasons
0,2013-14,5898,19,597,"{'Moong': 651, 'Other Pulses': 503, 'Maize': 4...","{'Total': 2187, 'Kharif': 1566, 'Rabi': 1473, ..."
1,2014-15,5622,19,560,"{'Moong': 606, 'Maize': 493, 'Other Pulses': 4...","{'Total': 2071, 'Kharif': 1507, 'Rabi': 1367, ..."
2,2015-16,5931,19,579,"{'Moong': 629, 'Other Pulses': 539, 'Maize': 5...","{'Total': 2145, 'Rabi': 1526, 'Kharif': 1518, ..."
3,2016-17,6423,19,611,"{'Moong': 677, 'Other Pulses': 611, 'Urad': 57...","{'Total': 2330, 'Kharif': 1750, 'Rabi': 1628, ..."
4,2017-18,6795,19,660,"{'Moong': 720, 'Other Pulses': 595, 'Urad': 59...","{'Total': 2403, 'Kharif': 1847, 'Rabi': 1677, ..."
5,2018-19,7469,19,668,"{'Moong': 829, 'Other Pulses': 683, 'Urad': 65...","{'Total': 2741, 'Kharif': 2107, 'Rabi': 1771, ..."
6,2019-20,7830,19,667,"{'Moong': 832, 'Other Pulses': 693, 'Urad': 66...","{'Total': 2850, 'Kharif': 2246, 'Rabi': 1885, ..."
7,2020-21,8101,19,675,"{'Moong': 868, 'Other Pulses': 708, 'Urad': 68...","{'Total': 2922, 'Kharif': 2293, 'Rabi': 2002, ..."
8,2021-22,7921,19,691,"{'Moong': 815, 'Other Pulses': 710, 'Urad': 63...","{'Total': 2849, 'Kharif': 2179, 'Rabi': 1998, ..."
9,2022-23,8389,19,715,"{'Moong': 909, 'Other Pulses': 733, 'Urad': 71...","{'Total': 3046, 'Kharif': 2373, 'Rabi': 2084, ..."


In [14]:
# Cell 15: Classify zero-area, positive-yield records.
# This is a diagnostic only. No records are deleted or modified.

classification_results = []
flagged_samples = []

for year in year_sequence:
    area_col = f"Area-{year}"
    production_col = f"Production-{year}"
    yield_col = f"Yield-{year}"

    area = pd.to_numeric(apy_historical[area_col], errors="coerce")
    production = pd.to_numeric(
        apy_historical[production_col], errors="coerce"
    )
    reported_yield = pd.to_numeric(
        apy_historical[yield_col], errors="coerce"
    )

    flagged = area.eq(0) & reported_yield.gt(0)

    classification_results.append({
        "year": year,
        "zero_area_zero_production_positive_yield": int(
            (flagged & production.eq(0)).sum()
        ),
        "zero_area_positive_production_positive_yield": int(
            (flagged & production.gt(0)).sum()
        ),
        "zero_area_missing_production_positive_yield": int(
            (flagged & production.isna()).sum()
        ),
        "zero_area_negative_production_positive_yield": int(
            (flagged & production.lt(0)).sum()
        ),
    })

    # Save a small, representative sample for manual inspection.
    sample = apy_historical.loc[
        flagged,
        ["State", "District", "Crop", "Season",
         area_col, production_col, yield_col]
    ].head(10).copy()

    sample.insert(0, "year", year)
    flagged_samples.append(sample)

classification_report = pd.DataFrame(classification_results)
flagged_record_samples = pd.concat(flagged_samples, ignore_index=True)

display(classification_report)

,year,zero_area_zero_production_positive_yield,zero_area_positive_production_positive_yield,zero_area_missing_production_positive_yield,zero_area_negative_production_positive_yield
0,2013-14,5296,602,0,0
1,2014-15,5068,554,0,0
2,2015-16,5338,593,0,0
3,2016-17,5768,655,0,0
4,2017-18,6011,784,0,0
5,2018-19,6700,769,0,0
6,2019-20,6993,837,0,0
7,2020-21,7193,908,0,0
8,2021-22,7025,896,0,0
9,2022-23,7471,918,0,0


In [15]:
# Cell 16: Inspect sample records without altering the source data.

display(flagged_record_samples.head(30))

print("Sample rows:", len(flagged_record_samples))
print("Source historical dataset shape:", apy_historical.shape)

,year,State,District,Crop,Season,Area-2013-14,Production-2013-14,Yield-2013-14,Area-2014-15,Production-2014-15,Yield-2014-15,Area-2015-16,Production-2015-16,Yield-2015-16,Area-2016-17,Production-2016-17,Yield-2016-17,Area-2017-18,Production-2017-18,Yield-2017-18,Area-2018-19,Production-2018-19,Yield-2018-19,Area-2019-20,Production-2019-20,Yield-2019-20,Area-2020-21,Production-2020-21,Yield-2020-21,Area-2021-22,Production-2021-22,Yield-2021-22,Area-2022-23,Production-2022-23,Yield-2022-23,Area-2023-24,Production-2023-24,Yield-2023-24,Area-2024-25,Production-2024-25,Yield-2024-25
0,2013-14,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,0.0,0.0,2377.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-14,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,0.0,0.0,526.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-14,Andaman And Nicobar Islands,Nicobars,Cereals,Total,0.0,0.0,923.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-14,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,0.0,0.0,526.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-14,Andaman And Nicobar Islands,Nicobars,Maize,Total,0.0,0.0,526.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2013-14,Andaman And Nicobar Islands,Nicobars,Nutri/Coarse Cereals,Rabi,0.0,0.0,526.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2013-14,Andaman And Nicobar Islands,Nicobars,Nutri/Coarse Cereals,Total,0.0,0.0,526.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2013-14,Andaman And Nicobar Islands,Nicobars,Other Pulses,Rabi,0.0,0.0,3865.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,2013-14,Andaman And Nicobar Islands,Nicobars,Other Pulses,Total,0.0,0.0,3865.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,2013-14,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,0.0,0.0,2377.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Sample rows: 120
Source historical dataset shape: (31356, 40)


In [16]:
# Cell 17: Create a compact, year-specific view of suspicious records.
# The original historical dataset remains unchanged.

sample_rows = []

for year in year_sequence:
    area_col = f"Area-{year}"
    production_col = f"Production-{year}"
    yield_col = f"Yield-{year}"

    area = pd.to_numeric(apy_historical[area_col], errors="coerce")
    production = pd.to_numeric(
        apy_historical[production_col], errors="coerce"
    )
    reported_yield = pd.to_numeric(
        apy_historical[yield_col], errors="coerce"
    )

    # Investigate records where reported yield is positive but area is zero.
    mask = area.eq(0) & reported_yield.gt(0)

    flagged = apy_historical.loc[
        mask, ["State", "District", "Crop", "Season"]
    ].copy()

    flagged["year"] = year
    flagged["area"] = area.loc[mask].values
    flagged["production"] = production.loc[mask].values
    flagged["reported_yield"] = reported_yield.loc[mask].values

    flagged["diagnostic_category"] = np.select(
        [
            flagged["production"].eq(0),
            flagged["production"].gt(0),
            flagged["production"].isna(),
        ],
        [
            "Zero area and zero production",
            "Zero area but positive production",
            "Zero area and missing production",
        ],
        default="Other"
    )

    sample_rows.append(flagged)

flagged_records_tidy = pd.concat(sample_rows, ignore_index=True)

# Show examples from both major categories, without duplicating the wide dataset.
for category in [
    "Zero area and zero production",
    "Zero area but positive production",
]:
    print(f"\n{category}")
    display(
        flagged_records_tidy.loc[
            flagged_records_tidy["diagnostic_category"].eq(category)
        ].head(10)
    )


Zero area and zero production


,State,District,Crop,Season,year,area,production,reported_yield,diagnostic_category
0,Andaman And Nicobar Islands,Nicobars,Cereals,Kharif,2013-14,0.0,0.0,2377.0,Zero area and zero production
1,Andaman And Nicobar Islands,Nicobars,Cereals,Rabi,2013-14,0.0,0.0,526.0,Zero area and zero production
2,Andaman And Nicobar Islands,Nicobars,Cereals,Total,2013-14,0.0,0.0,923.0,Zero area and zero production
3,Andaman And Nicobar Islands,Nicobars,Maize,Rabi,2013-14,0.0,0.0,526.0,Zero area and zero production
4,Andaman And Nicobar Islands,Nicobars,Maize,Total,2013-14,0.0,0.0,526.0,Zero area and zero production
5,Andaman And Nicobar Islands,Nicobars,Nutri/Coarse Cereals,Rabi,2013-14,0.0,0.0,526.0,Zero area and zero production
6,Andaman And Nicobar Islands,Nicobars,Nutri/Coarse Cereals,Total,2013-14,0.0,0.0,526.0,Zero area and zero production
7,Andaman And Nicobar Islands,Nicobars,Other Pulses,Rabi,2013-14,0.0,0.0,3865.0,Zero area and zero production
8,Andaman And Nicobar Islands,Nicobars,Other Pulses,Total,2013-14,0.0,0.0,3865.0,Zero area and zero production
9,Andaman And Nicobar Islands,Nicobars,Rice,Kharif,2013-14,0.0,0.0,2377.0,Zero area and zero production



Zero area but positive production


,State,District,Crop,Season,year,area,production,reported_yield,diagnostic_category
24,Andaman And Nicobar Islands,North And Middle Andaman,Other Pulses,Rabi,2013-14,0.0,0.02,5630.0,Zero area but positive production
25,Andaman And Nicobar Islands,North And Middle Andaman,Other Pulses,Total,2013-14,0.0,0.02,5630.0,Zero area but positive production
30,Andaman And Nicobar Islands,South Andamans,Cereals,Kharif,2013-14,0.0,0.01,2496.0,Zero area but positive production
32,Andaman And Nicobar Islands,South Andamans,Cereals,Total,2013-14,0.0,0.01,2494.0,Zero area but positive production
39,Andaman And Nicobar Islands,South Andamans,Other Pulses,Rabi,2013-14,0.0,0.01,4600.0,Zero area but positive production
40,Andaman And Nicobar Islands,South Andamans,Other Pulses,Total,2013-14,0.0,0.01,4600.0,Zero area but positive production
41,Andaman And Nicobar Islands,South Andamans,Rice,Kharif,2013-14,0.0,0.01,2496.0,Zero area but positive production
42,Andaman And Nicobar Islands,South Andamans,Rice,Total,2013-14,0.0,0.01,2496.0,Zero area but positive production
43,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Kharif,2013-14,0.0,0.01,2496.0,Zero area but positive production
44,Andaman And Nicobar Islands,South Andamans,Total Food Grains,Rabi,2013-14,0.0,0.01,3714.0,Zero area but positive production


In [17]:
# Cell 18: Save audit reports for reproducibility.
# These files document findings; they do not alter the raw source files.

AUDIT_DIR.mkdir(parents=True, exist_ok=True)

classification_report.to_csv(
    AUDIT_DIR / "apy_zero_area_positive_yield_classification.csv",
    index=False
)

flagged_records_tidy.to_csv(
    AUDIT_DIR / "apy_zero_area_positive_yield_records.csv",
    index=False
)

print("Saved audit reports:")
print(AUDIT_DIR / "apy_zero_area_positive_yield_classification.csv")
print(AUDIT_DIR / "apy_zero_area_positive_yield_records.csv")

Saved audit reports:
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\reports\data_audit\apy_zero_area_positive_yield_classification.csv
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\reports\data_audit\apy_zero_area_positive_yield_records.csv


In [18]:
# Cell 19: Inspect precision and distribution of area and production.
# This checks whether small positive values occur near zero.

precision_summary = []

for year in year_sequence:
    area_col = f"Area-{year}"
    production_col = f"Production-{year}"

    area = pd.to_numeric(apy_historical[area_col], errors="coerce")
    production = pd.to_numeric(
        apy_historical[production_col], errors="coerce"
    )

    precision_summary.append({
        "year": year,
        "area_zero": int(area.eq(0).sum()),
        "area_missing": int(area.isna().sum()),
        "area_positive_below_0_01": int(
            (area.gt(0) & area.lt(0.01)).sum()
        ),
        "area_positive_below_0_1": int(
            (area.gt(0) & area.lt(0.1)).sum()
        ),
        "production_zero": int(production.eq(0).sum()),
        "production_missing": int(production.isna().sum()),
        "production_positive_below_0_01": int(
            (production.gt(0) & production.lt(0.01)).sum()
        ),
    })

precision_report = pd.DataFrame(precision_summary)
display(precision_report)

,year,area_zero,area_missing,area_positive_below_0_01,area_positive_below_0_1,production_zero,production_missing,production_positive_below_0_01
0,2013-14,6018,10428,0,6818,5866,10548,0
1,2014-15,5777,10001,0,7262,5669,10152,0
2,2015-16,6008,9733,0,7376,6126,9778,0
3,2016-17,6484,8867,0,7387,6356,8929,0
4,2017-18,6863,7970,0,7675,6538,8036,0
5,2018-19,7589,7743,0,7581,7255,7864,0
6,2019-20,7968,7052,0,7711,7566,7196,0
7,2020-21,8156,6965,0,7630,7732,6965,0
8,2021-22,7973,7331,0,7492,7585,7331,0
9,2022-23,8550,6173,0,7893,8177,6173,0
